In [2]:
# ============================================================
# BattingEdge V9.5 - RANDOM FOREST Model Training
# Features: 99 raw pose + 8 angles (NO velocities)
# Target: 83-86% accuracy
# ============================================================

# ✅ FIX: Import Pandas first to prevent circular import errors
import pandas as pd 
import numpy as np
import pickle
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib

# Random Forest Imports
from sklearn.ensemble import RandomForestClassifier

# ================= CONFIG =================
# ✅ FIXED PATH: Pointing to the correct features folder
FEATURE_DIR = Path(r"D:\Users\Anoshia\BattingEdge_FYP\features")
MODEL_DIR   = Path(r"D:\Users\Anoshia\BattingEdge_FYP\backend\models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Algorithm Name
ALGO_NAME = "random_forest"

# Hyperparameters
N_ESTIMATORS = 100  # Number of trees
MAX_DEPTH = None    # Let trees grow fully
RANDOM_STATE = 42
# ==========================================

print("="*70)
print(f"BATTINGEDGE V9.5 - MODEL TRAINING ({ALGO_NAME.upper()})")
print("="*70)
print()

# ================= LOAD DATA =================
print("📂 Loading data...")

try:
    X_train = np.load(FEATURE_DIR / "X_train.npy")
    y_train = np.load(FEATURE_DIR / "y_train.npy")
    X_val   = np.load(FEATURE_DIR / "X_val.npy")
    y_val   = np.load(FEATURE_DIR / "y_val.npy")
    X_test  = np.load(FEATURE_DIR / "X_test.npy")
    y_test  = np.load(FEATURE_DIR / "y_test.npy")

    with open(FEATURE_DIR / "classes.pkl", "rb") as f:
        CLASSES = pickle.load(f)
except FileNotFoundError as e:
    print(f"\n❌ CRITICAL ERROR: Could not find data files.")
    print(f"   Checked directory: {FEATURE_DIR}")
    print("   Please ensure feature extraction was successful.")
    raise e

num_classes = len(CLASSES)
N, T, F = X_train.shape

print(f"Train: {X_train.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Classes: {CLASSES}")
print(f"Features per frame: {F} (99 pose + 8 angles)")
print()

# ================= CRITICAL: SCALING =================
print("⚖️  Applying StandardScaler...")

scaler = StandardScaler()

# Fit on training data (flatten to 2D)
N_train = X_train.shape[0]
X_train_2d = X_train.reshape(N_train * T, F)
scaler.fit(X_train_2d)

def scale_data(X):
    N, T, F = X.shape
    X_2d = X.reshape(N * T, F)
    X_scaled = scaler.transform(X_2d)
    return X_scaled.reshape(N, T, F)

X_train = scale_data(X_train)
X_val   = scale_data(X_val)
X_test  = scale_data(X_test)

# Save scaler and classes with algorithm name
scaler_path = MODEL_DIR / f"scaler_V9_5_{ALGO_NAME}.pkl"
classes_path = MODEL_DIR / f"classes_V9_5_{ALGO_NAME}.pkl"

joblib.dump(scaler, scaler_path)
joblib.dump(CLASSES, classes_path)
print(f"✅ Scaler saved: {scaler_path.name}")
print(f"✅ Classes saved: {classes_path.name}")
print()

# ================= FLATTENING (REQUIRED FOR RF) =================
print("Flattening 3D sequential data for Random Forest (N, T*F)...")
# Reshape from (N, 50, 107) -> (N, 5350)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat   = X_val.reshape(X_val.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)
print(f"New Input Shape: {X_train_flat.shape}")
print()

# ================= CLASS WEIGHTS =================
print("⚖️  Computing class weights...")

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

for i, cls in enumerate(CLASSES):
    print(f"  {cls:15s}: {class_weight_dict[i]:.3f}")
print()

# ================= MODEL =================
print("🏗️  Building Random Forest model...")

model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    class_weight=class_weight_dict,
    random_state=RANDOM_STATE,
    n_jobs=-1,  # Use all CPU cores
    verbose=1
)

print(model)
print()

# ================= TRAINING =================
print("="*70)
print("🚀 STARTING TRAINING")
print("="*70)
print()

# Combine Train and Val for RF (It doesn't use validation set for stopping)
# This gives RF more data to learn from
X_full_train = np.concatenate((X_train_flat, X_val_flat), axis=0)
y_full_train = np.concatenate((y_train, y_val), axis=0)

model.fit(X_full_train, y_full_train)

print()
print("="*70)
print("✅ TRAINING COMPLETE")
print("="*70)
print()

# Save final model
final_model_path = MODEL_DIR / f"battingedge_V9_5_{ALGO_NAME}_best.pkl"
joblib.dump(model, final_model_path)
print(f"💾 Saved final model: {final_model_path.name}")
print()

# ================= EVALUATION =================
print("="*70)
print("📊 EVALUATING ON TEST SET")
print("="*70)
print()

# Predict
y_pred = model.predict(X_test_flat)

# Classification report
print("📋 CLASSIFICATION REPORT:")
print()
report = classification_report(y_test, y_pred, target_names=CLASSES, digits=3)
print(report)

report_path = MODEL_DIR / f"report_V9_5_{ALGO_NAME}.txt"
with open(report_path, "w") as f:
    f.write(report)
print(f"✅ Report saved: {report_path.name}")
print()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("🔢 CONFUSION MATRIX:")
print("   (Rows = True, Cols = Predicted)")
print()
print("        ", "  ".join([f"{cls[:4]:>4s}" for cls in CLASSES]))
for i, cls in enumerate(CLASSES):
    print(f"{cls[:8]:8s}", "  ".join([f"{cm[i,j]:4d}" for j in range(num_classes)]))
print()

# Per-class accuracy
print("📈 PER-CLASS ACCURACY:")
print()
for i, cls in enumerate(CLASSES):
    correct = cm[i, i]
    total = cm[i, :].sum()
    accuracy = (correct / total * 100) if total > 0 else 0
    print(f"   {cls:15s}: {correct:3d}/{total:3d} = {accuracy:5.1f}%")

overall_acc = np.trace(cm) / np.sum(cm) * 100
print()
print(f"   {'OVERALL':15s}: {np.trace(cm):3d}/{np.sum(cm):3d} = {overall_acc:5.2f}%")
print()

# Major confusions
print("🔍 MAJOR CONFUSIONS (>3 cases):")
print()
confusions = []
for i in range(num_classes):
    for j in range(num_classes):
        if i != j and cm[i, j] > 3:
            confusions.append((CLASSES[i], CLASSES[j], cm[i, j]))

if confusions:
    confusions.sort(key=lambda x: x[2], reverse=True)
    for true_cls, pred_cls, count in confusions:
        print(f"   {true_cls:15s} → {pred_cls:15s}: {count} cases")
else:
    print("   ✅ No major confusions!")
print()

# ================= VISUALIZATIONS =================
print("📊 Generating visualizations...")

# Confusion matrix heatmap
cm_path = MODEL_DIR / f"confusion_matrix_V9_5_{ALGO_NAME}.png"
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"Confusion Matrix - V9.5 ({ALGO_NAME})", fontsize=14, fontweight='bold')
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(cm_path, dpi=300)
print(f"   ✅ Saved: {cm_path.name}")
plt.close()

# Feature Importance Plot (Replaces Training History)
imp_path = MODEL_DIR / f"feature_importance_V9_5_{ALGO_NAME}.png"
importances = model.feature_importances_
plt.figure(figsize=(12, 4))
plt.plot(importances)
plt.xlabel("Feature Index (0-5350)")
plt.ylabel("Importance Score")
plt.title("Feature Importance Distribution")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(imp_path, dpi=300)
print(f"   ✅ Saved: {imp_path.name}")
plt.close()

print()

# ================= COMPARISON =================
print("="*70)
print(f"🎉 V9.5 ({ALGO_NAME}) TRAINING COMPLETE")
print("="*70)
print()

# Save metadata
metadata = {
    "version": "V9.5",
    "algorithm": ALGO_NAME,
    "features": "99 raw pose + 8 angles (no velocities) - RF Flattened",
    "classes": CLASSES,
    "test_accuracy": float(overall_acc),
    "per_class_accuracy": {
        CLASSES[i]: float((cm[i,i] / cm[i,:].sum() * 100) if cm[i,:].sum() > 0 else 0)
        for i in range(num_classes)
    },
    "hyperparameters": {
        "n_estimators": N_ESTIMATORS,
        "max_depth": "None"
    }
}

metadata_path = MODEL_DIR / f"metadata_V9_5_{ALGO_NAME}.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("📦 Saved artifacts:")
print(f"   1. Final model: {final_model_path.name}")
print(f"   2. Scaler: {scaler_path.name}")
print(f"   3. Classes: {classes_path.name}")
print(f"   4. Report: {report_path.name}")
print(f"   5. Confusion matrix: {cm_path.name}")
print(f"   6. Feature Importance: {imp_path.name}")
print(f"   7. Metadata: {metadata_path.name}")
print()

if overall_acc >= 83:
    print("✨ EXCELLENT RESULT! Ready for deployment.")
elif overall_acc >= 81:
    print("✅ GOOD RESULT! High accuracy.")
elif overall_acc >= 79:
    print("→ ACCEPTABLE. Stable performance.")
else:
    print("⚠️ Below target. Review feature extraction.")

print("="*70)

BATTINGEDGE V9.5 - MODEL TRAINING (RANDOM_FOREST)

📂 Loading data...
Train: 3007 samples
Val:   388 samples
Test:  378 samples
Classes: ['Cover Drive', 'Cut Shot', 'Defense', 'Pull Shot', 'Sweep Shot']
Features per frame: 107 (99 pose + 8 angles)

⚖️  Applying StandardScaler...
✅ Scaler saved: scaler_V9_5_random_forest.pkl
✅ Classes saved: classes_V9_5_random_forest.pkl

Flattening 3D sequential data for Random Forest (N, T*F)...
New Input Shape: (3007, 5350)

⚖️  Computing class weights...
  Cover Drive    : 1.004
  Cut Shot       : 1.018
  Defense        : 0.972
  Pull Shot      : 0.999
  Sweep Shot     : 1.009

🏗️  Building Random Forest model...
RandomForestClassifier(class_weight={0: 1.0040066777963272,
                                     1: 1.0175972927241963,
                                     2: 0.9715670436187399,
                                     3: 0.9990033222591362,
                                     4: 1.0090604026845638},
                       n_jobs=-1, random_

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    6.8s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   14.5s finished



✅ TRAINING COMPLETE

💾 Saved final model: battingedge_V9_5_random_forest_best.pkl

📊 EVALUATING ON TEST SET



[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


📋 CLASSIFICATION REPORT:

              precision    recall  f1-score   support

 Cover Drive      0.957     0.859     0.905        78
    Cut Shot      0.860     0.914     0.886        81
     Defense      0.882     0.893     0.887        75
   Pull Shot      0.912     0.873     0.892        71
  Sweep Shot      0.872     0.932     0.901        73

    accuracy                          0.894       378
   macro avg      0.897     0.894     0.894       378
weighted avg      0.896     0.894     0.894       378

✅ Report saved: report_V9_5_random_forest.txt

🔢 CONFUSION MATRIX:
   (Rows = True, Cols = Predicted)

         Cove  Cut   Defe  Pull  Swee
Cover Dr   67     3     3     1     4
Cut Shot    1    74     2     3     1
Defense     0     3    67     1     4
Pull Sho    2     5     1    62     1
Sweep Sh    0     1     3     1    68

📈 PER-CLASS ACCURACY:

   Cover Drive    :  67/ 78 =  85.9%
   Cut Shot       :  74/ 81 =  91.4%
   Defense        :  67/ 75 =  89.3%
   Pull Shot      :